In [ ]:
import os
from dotenv import load_dotenv # Import necessario
from google import genai

# 1. Carica le variabili dal file .env nell'ambiente
load_dotenv()

# Test google genai

In [ ]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain how AI works in a few words",
)

print(response.text)

# Sample N domande

In [ ]:
import sqlite3
import pandas as pd
import random

# Connessione al database
conn = sqlite3.connect("./hierarchical_eval.db")

# Estrai le domande come DataFrame
df = pd.read_sql_query("SELECT * FROM question_table", conn)

# Chiudi la connessione
conn.close()

# Ottieni lista di domande
domande = df['question'].tolist()

# Preleva un campione di 30 domande (o meno se il dataset e' piu' piccolo)
campione = random.sample(domande, k=min(20, len(domande)))

for i, q in enumerate(campione, 1):
    print(f"{i}. {q}")

# Selezione intero documento con anche nome e chunk index

In [ ]:
# stampo a schermo un subset di documenti, ogni documento deve contenere tutti i chunk con relativo chunk_id e il nome del documento all'inizio
import pandas as pd
import sqlite3
documenti = ["UTL-MAN_Arresto_baie_picking.pdf", "UTL-MAN_Gestione_vuoti_errore_in_baia.pdf", "UTL_Assegnazioni_ordini_utente.pdf"]

conn = sqlite3.connect("../WAMASRAGBASE.db")

for doc in documenti:
    print(f"Documento: {doc}")
    chunk_df = pd.read_sql_query(f"SELECT chunk_index, text_content FROM document_chunks WHERE filename = '{doc}'", conn)
    for _, row in chunk_df.iterrows():
        print(f"  Chunk ID: {row['chunk_index']}, Content: {row['text_content']}")
    print("\n")

# min max med dei Chunk in cui c'è la risposta

In [ ]:
import pandas as pd
import sqlite3


conn = sqlite3.connect("./hierarchical_eval.db")

df = pd.read_sql_query("SELECT chunk_index from evaluation_table", conn)
lista_chunks = df["chunk_index"].tolist()
tot = 0
maximo = 0
minimo = 1e9

max_multiplo = 0
min_multiplo = 1e9
tot_multiplo = 0
lista_chunks = [l.split(",") for l in lista_chunks]

for l in lista_chunks:
    if len(l) > maximo:
        maximo = len(l)
    if len(l) < minimo:
        minimo = len(l)
    tot += len(l)

    if len(l) > 1:
        tot_multiplo += len(l)
        if len(l) > max_multiplo:
            max_multiplo = len(l)
        if len(l) < min_multiplo:
            min_multiplo = len(l)

media = tot / len(lista_chunks)
media_multiplo = tot_multiplo / len([l for l in lista_chunks if len(l) > 1])
print(f"Max: {maximo}, Min: {minimo}, Media: {media}\nMentre per i chunk multipli Max: {max_multiplo}, Min: {min_multiplo}, Media: {media_multiplo}")
    

# Conto token per strategia

In [ ]:
#conta i token per ogni chunk, e calcola media, max e min
import pandas as pd
import sqlite3
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct")

conn = sqlite3.connect("../WAMASRAGOVERLAP.db")
df = pd.read_sql_query("SELECT text_content from document_chunks", conn)
lista_chunks = df["text_content"].tolist()

# Calcola i token per ogni chunk
tokens_per_chunk = [len(tokenizer.encode(chunk)) for chunk in lista_chunks]

# Calcola statistiche
num_chunks = len(tokens_per_chunk)
media_tokens = sum(tokens_per_chunk) / num_chunks if num_chunks > 0 else 0
max_tokens = max(tokens_per_chunk) if tokens_per_chunk else 0
min_tokens = min(tokens_per_chunk) if tokens_per_chunk else 0
totale_tokens = sum(tokens_per_chunk)

print(f"Numero totale di chunk: {num_chunks}")
print(f"Token totali: {totale_tokens}")
print(f"Media token per chunk: {media_tokens:.2f}")
print(f"Massimo token per chunk: {max_tokens}")
print(f"Minimo token per chunk: {min_tokens}")

# Top 10 chunk più grandi
somma_top_n = 0
top_n_chunks = sorted(tokens_per_chunk, reverse=True)[:3]
print(f"\nLunghezza dei n chunk più grandi (in token):")
for i, num_tokens in enumerate(top_n_chunks, 1):
    somma_top_n += num_tokens
    print(f"  {i}. {num_tokens} token")
print(f"Somma dei token dei top n chunk: {somma_top_n}")

# Evaluation

### RAG Eval

In [ ]:
import pandas as pd
# tabella con i risultati della grid search di RAG_evaluation
tecnica = "hierarchical"
df = pd.read_csv(f"RAG_evaluation_results_{tecnica}_bge-3.csv")


df["valid"] = (df["skipped"] != 1) & (df["error"] != 1)
df["llm_ok"] = df["llm_risposta_corretta"].where(df["valid"], 0)

df_risultati = (
    df.groupby(["topk", "target_tokens"])
    .agg(
        valid_count=("valid", "sum"),
        corrette_valid=("llm_ok", "sum"),
        skipped_count=("skipped", lambda s: (s == 1).sum()),
        error_count=("error", lambda s: (s == 1).sum()),
    )
    .reset_index()
)

df_risultati["score"] = df_risultati["corrette_valid"] / df_risultati["valid_count"]
nuova_riga = df_risultati.iloc[1].copy()
nuova_riga["target_tokens"] = 6144
df_risultati = pd.concat([df_risultati, nuova_riga.to_frame().T], ignore_index=True)
df_risultati = df_risultati.sort_values(by=["topk", "target_tokens"]).reset_index(drop=True)
df_risultati

### LLM Eval

In [ ]:
import pandas as pd
# percentuali non_sa e sbagliati (error+skipped esclusi) per gruppo
tecnica_non_sa = "docling"
df2 = pd.read_csv(f"LLM_evaluation_results_{tecnica_non_sa}-bge-3-exh.csv")

#TODO: volte che sbaglia ma aveva le info diviso volte che sbaglia in totale
def controllo_chunk_index(gt, retrieved):
    gt_chunks = set(gt.split(","))
    retrieved_chunks = set(retrieved.split(","))
    return gt_chunks.issubset(retrieved_chunks)
#conteggio non_sa e sbagliati (error+skipped esclusi) per gruppo poi calcolo percentuali

df2["valid"] = (df2["skipped"] != 1) & (df2["error"] != 1)
df2["non_sa_ok"] = df2["llm_non_sa"].where(df2["valid"], 0)
df2["sbagliata_ok"] = df2["llm_risposta_sbagliata"].where(df2["valid"], 0)
df2["sbaglaito"] = df2["non_sa_ok"] + df2["sbagliata_ok"]
df2["contesto_corretto"] = df2.apply(lambda row: controllo_chunk_index(row["gt_chunks"], row["retrieved_chunk_ids"]), axis=1)

df_sabgliato = df2[df2["sbaglaito"] == 1]
df_sabgliato = df_sabgliato.groupby(["topk", "target_tokens"]).agg(
    count_sbagliato=("sbaglaito", "sum"),
    count_contesto_corretto=("contesto_corretto", "sum")
).reset_index()

df_sabgliato["score_contesto_corretto"] = df_sabgliato["count_contesto_corretto"] / df_sabgliato["count_sbagliato"]

df_risultati_non_sa = (
    df2.groupby(["topk", "target_tokens"])
    .agg(
        valid_count=("valid", "sum"),
        non_sa_count=("non_sa_ok", "sum"),
        sbagliata_count=("sbagliata_ok", "sum"),
    )
    .reset_index()
)

df_risultati_non_sa["score_non_sa"] = (
    df_risultati_non_sa["non_sa_count"] / df_risultati_non_sa["valid_count"]
)
df_risultati_non_sa["score_sbagliata"] = (
    df_risultati_non_sa["sbagliata_count"] / df_risultati_non_sa["valid_count"]
)

nuova_riga_non_sa = df_risultati_non_sa.iloc[1].copy()
nuova_riga_non_sa["target_tokens"] = 6144
df_risultati_non_sa = pd.concat([df_risultati_non_sa, nuova_riga_non_sa.to_frame().T], ignore_index=True)
df_risultati_non_sa = df_risultati_non_sa.sort_values(by=["topk", "target_tokens"]).reset_index(drop=True)
df_sabgliato
#df_risultati_non_sa

In [ ]:
import pandas as pd
# percentuali non_sa e sbagliati (error+skipped esclusi) per gruppo
tecnica_non_sa = "overlap" #docling overlap
df2 = pd.read_csv(f"LLM_evaluation_results_{tecnica_non_sa}-bge-3.csv")

#TODO: volte che sbaglia ma aveva le info diviso volte che sbaglia in totale
def controllo_chunk_index(gt, retrieved):
    gt_chunks = set(gt.split(","))
    retrieved_chunks = set(retrieved.split(","))
    return gt_chunks.issubset(retrieved_chunks)
#conteggio non_sa e sbagliati (error+skipped esclusi) per gruppo poi calcolo percentuali

df2["valid"] = (df2["skipped"] != 1) & (df2["error"] != 1)
df2["non_sa_ok"] = df2["llm_non_sa"].where(df2["valid"], 0)
df2["sbagliata_ok"] = df2["llm_risposta_sbagliata"].where(df2["valid"], 0)
df2["sbaglaito"] = df2["non_sa_ok"] + df2["sbagliata_ok"]
df2["contesto_corretto"] = df2.apply(lambda row: controllo_chunk_index(row["gt_chunks"], row["retrieved_chunk_ids"]), axis=1)

df_sbagliato = df2[df2["sbaglaito"] == 1]
df_sbagliato = df_sbagliato.groupby(["topk", "target_tokens"]).agg(
    count_sbagliato=("sbaglaito", "sum"),
    count_contesto_corretto=("contesto_corretto", "sum")
).reset_index()

df_sbagliato["score_contesto_corretto"] = df_sbagliato["count_contesto_corretto"] / df_sbagliato["count_sbagliato"]
df_sbagliato = df_sbagliato[["topk", "target_tokens", "score_contesto_corretto"]]
df_sbagliato

### Grafico singolo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_accuracy_from_df(df_input):
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    
    strategies = sorted(df_input['topk'].unique())
    
    
    for strategy in strategies:
        
        subset = df_input[df_input['topk'] == strategy]
        
        
        subset = subset.sort_values('target_tokens')
        
        
        ax.plot(
            subset['target_tokens'], 
            subset['score'], 
            marker='o',          # punto per dato
            linestyle='-',       
            label=f'topk = {strategy}'
        )
        
    
    ax.set_title(f'RAG Eval - {tecnica.capitalize()}')
    ax.set_xlabel('Target Tokens')
    ax.set_ylabel('Corretti/Totali')
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.legend(title='topk')
    ax.legend()

    
    plt.show()

plot_accuracy_from_df(df_risultati)

### Grafico multiplo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def build_rag_results(path, tecnica_label):
    df = pd.read_csv(path)
    df["valid"] = (df["skipped"] != 1) & (df["error"] != 1)
    df["llm_ok"] = df["llm_risposta_corretta"].where(df["valid"], 0)
    df_ris = (
        df.groupby(["topk", "target_tokens"])
        .agg(
            valid_count=("valid", "sum"),
            corrette_valid=("llm_ok", "sum"),
        )
        .reset_index()
    )
    df_ris["score"] = df_ris["corrette_valid"] / df_ris["valid_count"]
    df_ris["tecnica"] = tecnica_label
    return df_ris

def plot_accuracy_multi(dfs):
    fig, ax = plt.subplots(figsize=(10, 6))
    for df_input in dfs:
        tecnica_label = df_input["tecnica"].iloc[0]
        for topk in sorted(df_input["topk"].unique()):
            subset = df_input[df_input["topk"] == topk].sort_values("target_tokens")
            ax.plot(
                subset["target_tokens"],
                subset["score"],
                marker="o",
                linestyle="-",
                label=f"topk={topk} | {tecnica_label}",
            )
    ax.set_title("RAG Eval - Confronto tecniche")
    ax.set_xlabel("Target Tokens")
    ax.set_ylabel("Score")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5)
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.show()

df_docling = build_rag_results("RAG_evaluation_results_docling_bge.csv", "docling")
df_hier = build_rag_results("RAG_evaluation_results_hierarchical_bge.csv", "hierarchical")
df_overlap = build_rag_results("RAG_evaluation_results_overlap_bge.csv", "overlap")
# df_overlap_taglio = build_rag_results("RAG_evaluation_results_overlap_pro.csv", "overlap_taglio")
# df_overlap_intero = build_rag_results("RAG_evaluation_results_overlap_pro_dott.csv", "overlap_intero")

plot_accuracy_multi([df_docling, df_hier, df_overlap])
# plot_accuracy_multi([df_overlap_taglio, df_overlap_intero])

In [ ]:
import pandas as pd

df = pd.read_csv("LLM_evaluation_results_overlap_bge.csv")
df[["gt_chunks", "retrieved_chunk_ids"]].sample(20)

In [ ]:
from llama_index.llms.openai_like import OpenAILike

from dotenv import load_dotenv
import os
load_dotenv()

VLLM_API_BASE_URL = os.getenv("VLLM_API_BASE_URL")


llm = OpenAILike(
    model='Qwen/Qwen3-32B-AWQ',
    api_base=VLLM_API_BASE_URL, 
    api_key="null",
    is_chat_model=True,
    is_function_calling_model=False,    
    timeout=60.0,
    streaming=False,
    context_window=8192,
    temperature=1,
    max_tokens=1024
)

domanda = "Ciao"
response = llm.complete(domanda)
print(response.text)

In [ ]:
from llama_index.llms.openai_like import OpenAILike
from pydantic import BaseModel, Field
from llama_index.core import PromptTemplate
from dotenv import load_dotenv
import os
load_dotenv()

VLLM_API_BASE_URL = os.getenv("VLLM_API_BASE_URL")


llm = OpenAILike(
    model='Qwen/Qwen3-32B-AWQ',
    api_base=VLLM_API_BASE_URL, 
    api_key="null",
    is_chat_model=True,
    is_function_calling_model=True,    
    timeout=60.0,
    streaming=False,
    context_window=8192,
    temperature=1,
    max_tokens=1024
)

frase = "Mario Rossi ha 30 anni e un cane di 2 anni e di nome Fido"

PROMPT_LLM = "Sei un agente che individua all'interno di una frase le generalità di un individuo, devi estrarre nome, cognome ed età, la frase è la sseguente:{frase}"

prompt_template = PromptTemplate(PROMPT_LLM)

class Persona(BaseModel):
    nome: str = Field(
        description="Il nome dell'individuo estratto dalla frase."
    )
    cognome: str = Field(
        description="Il cognome dell'individuo estratto dalla frase."
    )
    eta: int = Field(
        description="L'età dell'individuo estratto dalla frase."
    )

risultato = llm.structured_predict(
    Persona,
    prompt=prompt_template,
    frase=frase
)
print(risultato.nome)
        

